In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
import pickle

In [2]:
df = pd.read_csv('/content/heart_disease_synthetic_25000.csv')

In [3]:
df

,patient_id,age,sex,chest_pain_type,resting_bp,cholesterol,fasting_blood_sugar_gt_120,resting_ecg,max_heart_rate,exercise_angina,...,family_history,physical_activity,alcohol_use,sleep_hours,stress_level,creatinine,hemoglobin,hs_crp,troponin,heart_disease
0,PT000001,58,Male,Non_anginal,126.0,145.0,0,ST-T_abnormality,154,1,...,0,High,Occasional,6.6,Moderate,0.86,16.8,0.79,0.061,0
1,PT000002,42,Male,Typical_angina,119.0,197.0,0,Normal,151,0,...,0,Moderate,NaN,7.7,Low,1.04,12.7,2.12,0.018,0
2,PT000003,63,Female,Non_anginal,134.0,229.0,0,ST-T_abnormality,150,0,...,0,High,Frequent,6.2,Low,0.99,14.8,2.05,0.040,0
3,PT000004,65,Female,Asymptomatic,157.0,181.0,0,ST-T_abnormality,125,0,...,0,High,Frequent,8.6,Moderate,0.86,13.2,3.34,0.493,0
4,PT000005,31,Female,Non_anginal,136.0,193.0,0,Normal,157,0,...,1,High,Occasional,6.9,Low,1.17,13.3,2.51,0.073,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,PT024996,68,Male,Non_anginal,123.0,204.0,1,Normal,129,1,...,0,Moderate,NaN,8.1,Moderate,1.41,14.3,0.75,0.339,0
24996,PT024997,49,Male,Atypical_angina,117.0,194.0,1,Normal,109,1,...,0,Moderate,Occasional,5.5,High,0.99,15.0,2.02,0.062,1
24997,PT024998,54,Male,Non_anginal,125.0,231.0,0,Normal,98,0,...,0,Moderate,NaN,8.2,High,0.70,15.8,1.85,0.089,1
24998,PT024999,47,Female,Non_anginal,130.0,228.0,0,Normal,96,1,...,0,Low,Frequent,7.4,High,1.64,13.8,3.02,0.251,1


In [4]:
df.isnull().sum()

,0
patient_id,0
age,0
sex,0
chest_pain_type,0
resting_bp,259
cholesterol,460
fasting_blood_sugar_gt_120,0
resting_ecg,295
max_heart_rate,0
exercise_angina,0


In [5]:
df.loc[df['resting_bp'].isnull(),'resting_bp']=df['resting_bp'].median()

In [6]:
df.loc[df['cholesterol'].isnull(),'cholesterol']=df['cholesterol'].median()

In [7]:
df.loc[df['resting_ecg'].isnull(), 'resting_ecg'] = 'Unknown'

In [8]:
df.thalassemia.value_counts()

,count
thalassemia,
Normal,13770
Reversible_defect,6590
Fixed_defect,4141


In [9]:
df.loc[df['thalassemia'].isnull(), 'thalassemia'] = 'Unknown'

In [10]:
df.loc[df['alcohol_use'].isnull(),'alcohol_use'] = 'Unknown'

In [11]:
df.loc[df['sleep_hours'].isnull(),'sleep_hours'] = df['sleep_hours'].median()

In [12]:
df.loc[df['creatinine'].isnull(),'creatinine'] = df['creatinine'].median()

In [13]:
df.loc[df['hemoglobin'].isnull(),'hemoglobin'] = df['hemoglobin'].median()

In [14]:
df.loc[df['hs_crp'].isnull(),'hs_crp'] = df['hs_crp'].median()

In [15]:
df.loc[df['bmi'].isnull(),'bmi'] = df['bmi'].median()

In [16]:
df.isnull().sum()

,0
patient_id,0
age,0
sex,0
chest_pain_type,0
resting_bp,0
cholesterol,0
fasting_blood_sugar_gt_120,0
resting_ecg,0
max_heart_rate,0
exercise_angina,0


In [17]:
with open('/content/pipeline (3).pkl','rb') as f:
  pipeline=pickle.load(f)

In [18]:
pipeline

ColumnTransformer(transformers=[('OHE_columns', OneHotEncoder(),
                                 ['sex', 'chest_pain_type', 'resting_ecg',
                                  'thalassemia']),
                                ('Ordinal_st_slope',
                                 OrdinalEncoder(categories=[['Upsloping',
                                                             'Flat',
                                                             'Downsloping']]),
                                 ['st_slope']),
                                ('Ordinal_smoking_status',
                                 OrdinalEncoder(categories=[['Never', 'Former',
                                                             'Current']]),
                                 ['smoking_status']),
                                ('Ordinal_physical_activity',
                                 O...
                                ('Ordinal_stress_level',
                                 OrdinalEncoder(categories=[['Moderate', 'Low',
                                                             'High']]),
                                 ['stress_level']),
                                ('passthrough', 'passthrough',
                                 ['age', 'resting_bp', 'cholesterol',
                                  'fasting_blood_sugar_gt_120',
                                  'num_major_vessels', 'max_heart_rate',
                                  'exercise_angina', 'st_depression', 'bmi',
                                  'diabetes', 'hypertension', 'family_history',
                                  'sleep_hours', 'creatinine', 'hemoglobin',
                                  'hs_crp', 'troponin'])])

In [19]:
x = df.drop(columns=['heart_disease', 'patient_id'])
y = df['heart_disease']

In [20]:
from sklearn.model_selection import  train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.25,random_state=33)

In [35]:
for name, transformer, columns in pipeline.transformers:
    if name == 'OHE_columns':
        transformer.handle_unknown = 'ignore'

x_train_transformed = pipeline.fit_transform(x_train)
x_test_transformed = pipeline.transform(x_test)

In [36]:
x_train_transformed

array([[ 1.   ,  0.   ,  0.   , ..., 12.4  ,  0.68 ,  0.06 ],
       [ 0.   ,  1.   ,  0.   , ..., 15.   ,  0.93 ,  0.041],
       [ 1.   ,  0.   ,  0.   , ..., 11.8  ,  1.35 ,  0.241],
       ...,
       [ 1.   ,  0.   ,  0.   , ..., 13.6  ,  0.93 ,  0.168],
       [ 0.   ,  1.   ,  0.   , ..., 12.   ,  2.22 ,  0.158],
       [ 1.   ,  0.   ,  0.   , ..., 14.4  ,  0.65 ,  0.061]],
      shape=(18750, 36))

In [29]:
from sklearn.ensemble import GradientBoostingClassifier
gbm=GradientBoostingClassifier()
gbm.fit(x_train_transformed,y_train)

GradientBoostingClassifier()

In [30]:
x_test_transformed = pipeline.transform(x_test)

In [31]:
y_gbm=gbm.predict(x_test_transformed)
y_gbm

array([0, 0, 0, ..., 0, 1, 0], shape=(6250,))

In [32]:
from sklearn.metrics import classification_report ,accuracy_score,f1_score

In [33]:
print(accuracy_score(y_test,y_gbm))

0.70272


In [34]:
from sklearn.model_selection import RandomizedSearchCV


XGB=GradientBoostingClassifier()
param_grid = {
    'learning_rate': [0.01, 0.03, 0.06, 0.1, 0.4, 0.5, 0.6, 0.7],
    'max_depth': [5, 6, 7, 8, 9, 10],
    'n_estimators': [50, 65, 80, 100],
}

rf_cv = RandomizedSearchCV(
    estimator=XGB, scoring='f1',refit = True,param_distributions=param_grid, cv=5,
                               verbose=2
)
rf_cv.fit(x_train_transformed,y_train)
cv_best_params = rf_cv.best_params_

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END ....learning_rate=0.4, max_depth=7, n_estimators=65; total time=   8.3s
[CV] END ....learning_rate=0.4, max_depth=7, n_estimators=65; total time=   7.2s
[CV] END ....learning_rate=0.4, max_depth=7, n_estimators=65; total time=   9.3s
[CV] END ....learning_rate=0.4, max_depth=7, n_estimators=65; total time=   8.3s
[CV] END ....learning_rate=0.4, max_depth=7, n_estimators=65; total time=   7.3s
[CV] END ....learning_rate=0.1, max_depth=9, n_estimators=65; total time=  11.7s
[CV] END ....learning_rate=0.1, max_depth=9, n_estimators=65; total time=  11.6s
[CV] END ....learning_rate=0.1, max_depth=9, n_estimators=65; total time=  11.6s
[CV] END ....learning_rate=0.1, max_depth=9, n_estimators=65; total time=  11.4s
[CV] END ....learning_rate=0.1, max_depth=9, n_estimators=65; total time=  18.1s
[CV] END ....learning_rate=0.6, max_depth=5, n_estimators=50; total time=   3.8s
[CV] END ....learning_rate=0.6, max_depth=5, n_e

In [37]:
print(f"Best paramters: {cv_best_params})")

Best paramters: {'n_estimators': 65, 'max_depth': 5, 'learning_rate': 0.06})


In [38]:
XGB2=GradientBoostingClassifier(n_estimators=65, max_depth=5, learning_rate=0.06)
XGB2.fit(x_train_transformed,y_train)

GradientBoostingClassifier(learning_rate=0.06, max_depth=5, n_estimators=65)

In [42]:
y_predict=XGB2.predict(x_test_transformed)

In [40]:
print(accuracy_score(y_test,y_predict))

0.7016


In [43]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier()
rf_clf.fit(x_train_transformed,y_train)

RandomForestClassifier()

In [44]:
y_predict=rf_clf.predict(x_test_transformed)

In [45]:
y_predict

array([0, 0, 0, ..., 0, 1, 0], shape=(6250,))

In [46]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_predict)

0.69936

In [48]:
from xgboost import XGBClassifier
xgb_r=XGBClassifier()
xgb_r.fit(x_train_transformed,y_train)
y_hat=xgb_r.predict(x_test_transformed)

In [49]:
print(accuracy_score(y_hat,y_test))

0.6944
